In [ ]:
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# ----------------------------
# Configuration
# ----------------------------
input_csv = "./data/BioBERT_NER/preprocessed_data_ner.csv"   # 🔹 Your input CSV
text_column = "Preprocessed Posts"               # 🔹 Text column name
output_csv = input_csv.replace(".csv", "_with_entities.csv")

MODEL = "d4data/biomedical-ner-all"  # Good biomedical NER model

# ----------------------------
# Load model and tokenizer
# ----------------------------
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForTokenClassification.from_pretrained(MODEL)
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)

# ----------------------------
# Helper: Chunk long text safely
# ----------------------------
def chunk_text(text, max_tokens=400):
    """Splits long text into manageable chunks for BioBERT."""
    words = text.split()
    chunks, current = [], []
    for word in words:
        current.append(word)
        if len(current) >= max_tokens:
            chunks.append(" ".join(current))
            current = []
    if current:
        chunks.append(" ".join(current))
    return chunks

# ----------------------------
# Load data
# ----------------------------
df = pd.read_csv(input_csv)
if text_column not in df.columns:
    raise ValueError(f"Column '{text_column}' not found in CSV. Available columns: {list(df.columns)}")

# ----------------------------
# Extract entities
# ----------------------------
entities_list = []
print("Extracting NER entities from long texts...")

for text in tqdm(df[text_column].astype(str).tolist()):
    try:
        all_entities = []
        chunks = chunk_text(text)
        for chunk in chunks:
            ner_results = ner_pipeline(chunk)
            chunk_entities = [f"{ent['word']} ({ent['entity_group']})" for ent in ner_results]
            all_entities.extend(chunk_entities)
        entities_list.append("; ".join(all_entities))
    except Exception as e:
        print(f"Error processing text: {text[:80]}... -> {e}")
        entities_list.append("")

# ----------------------------
# Save output
# ----------------------------
df["Entities"] = entities_list
df.to_csv(output_csv, index=False)
print(f"\n✅ Done! Results saved to: {output_csv}")
